# Using LODE from Python

Read an OWL or SKOS artefact, walk its entities, serialize it and build a static site. Install with `pip install oc-lode`.

Paths below are relative to this notebook, inside the LODE repository.

## 1. Read an artefact

Each format has its own class: `OwlOntology`, `SkosVocabulary`. The source is a local path or a public http(s) URL. Pass a URL as a plain string, never as a `Path`: on Windows a `Path` rewrites the slashes and the URL is no longer valid.

In [1]:
from lode import OwlOntology

# load an ontology located on the web
source = "http://purl.org/spar/cito/"
ontology = OwlOntology(source)

print(ontology.read_as, ontology.artefact_iri)

owl http://purl.org/spar/cito


In [2]:
import tempfile
from pathlib import Path

from lode import SkosVocabulary

# inline Turtle saved to a temp file, so the reader can load it from a local path
turtle = """
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex: <http://example.org/animals/> .

ex:scheme a skos:ConceptScheme ;
    skos:prefLabel "Animals"@en .

ex:animal a skos:Concept ;
    skos:prefLabel "animal"@en ;
    skos:inScheme ex:scheme ;
    skos:topConceptOf ex:scheme .

ex:cat a skos:Concept ;
    skos:prefLabel "cat"@en ;
    skos:broader ex:animal ;
    skos:inScheme ex:scheme .
"""

# load a skos vocabulary from local path
path = Path(tempfile.mkdtemp(prefix="lode_")) / "animals.ttl"
path.write_text(turtle, encoding="utf-8")

vocabulary = SkosVocabulary(str(path))
print(vocabulary.read_as, vocabulary.artefact_iri)

skos http://example.org/animals/scheme


When the format is only known as a string, use `lode.read`. It returns the same class.

In [3]:
import lode 

## ggiungi clss
owl = lode.read(source=source, read_as="owl")
skos = lode.read(source=str(path), read_as="skos")

print(type(owl).__name__, owl.read_as)
print(type(skos).__name__, skos.read_as)

OwlOntology owl
SkosVocabulary skos


## 2. The model

An artefact is made of **model objects**: instances of the classes in `lode.models`, the same for every format. The reader fills them, the viewers read them.

### 2.1 The classes

Almost every class extends `Resource`. `Literal` is the exception: it is a value, not a resource.

In [4]:
import lode.models as models


def show_tree(cls, depth=0):
    print("  " * depth + cls.__name__)
    for sub in cls.__subclasses__():
        show_tree(sub, depth + 1)


show_tree(models.Resource)
print(models.Literal.__name__, "(not a Resource)")

Resource
  Property
    Relation
    Annotation
    Attribute
  Concept
    Datatype
    Restriction
      TruthFunction
      OneOf
      PropertyConceptRestriction
        Quantifier
        Cardinality
      Value
      DatatypeRestriction
      PropertySelfRestriction
  Model
  Individual
  Statement
  Container
  Collection
  Variable
  Atom
  Rule
Literal (not a Resource)


### 2.2 Which classes each format produces

In OWL, `Concept` is a class, `Relation` an object property, `Attribute` a datatype property, `Annotation` an annotation property, `Individual` a named individual and `Model` the ontology itself; `Datatype` and the restrictions are special concepts. In SKOS the classes are `Concept`, `Collection` and `Model` (the concept scheme).

The entities of the two artefacts read above, counted by class:

In [5]:
from collections import Counter

print("OWL: ", Counter(type(i).__name__ for i in ontology.instances()))
print("SKOS:", Counter(type(i).__name__ for i in vocabulary.instances()))

OWL:  Counter({'Statement': 311, 'Relation': 101, 'Annotation': 36, 'Resource': 13, 'Concept': 11, 'Model': 4, 'Cardinality': 4, 'Datatype': 4, 'Attribute': 3, 'TruthFunction': 1})
SKOS: Counter({'Concept': 2, 'Model': 1})


### 2.3 Fields and relations

A model class holds **fields**: plain data (`has_identifier`, `is_deprecated`) and relations to other model objects (`has_label` points to literals, `is_sub_concept_of` to concepts). Each field `x` is read with `get_x()` and written with `set_x()`: `has_label` becomes `get_has_label()`. These are also the names a profile's `reader.yaml` uses to fill the model, for example `set_has_exact_match`.

Python's `vars` lists the fields of a class even before you have any entity:

In [6]:
from lode.models import Concept, Literal

print(list(vars(Concept())))   # what a Concept can hold
print(list(vars(Literal())))   # what a Literal can hold

['has_identifier', 'is_deprecated', 'has_comment', 'has_label', 'has_preferred_label', 'has_alternative_label', 'has_hidden_label', 'has_notation', 'has_note', 'has_change_note', 'has_definition', 'has_editorial_note', 'has_example', 'has_history_note', 'has_scope_note', 'see_also', 'is_defined_by', 'has_version_info', 'also_defined_as', 'has_type', 'is_included_in', 'is_in_range_of', 'is_sub_concept_of', 'is_super_concept_of', 'is_disjoint_with', 'is_equivalent_to', 'is_related_to', 'has_broad_match', 'has_narrow_match', 'has_related_match', 'has_exact_match', 'has_close_match', 'is_top_concept_of', 'has_key', 'individuals_with_this_type', 'is_in_domain_of']
['has_language', 'has_value', 'has_type']


Most fields are empty for a given entity. This keeps the ones that hold a value; the type of each value tells which class the relation points to. An OWL class first:

In [7]:
owl_concept = ontology.instances(Concept)[0]

print(owl_concept)
{name: value for name, value in vars(owl_concept).items() if value}

Concept('http://purl.org/spar/cito/JournalCartelCitation')


{'has_identifier': 'http://purl.org/spar/cito/JournalCartelCitation',
 'has_comment': [Literal('A citation from one journal to another journal which forms one of a very large number of citations from the citing journal to recent articles in the cited journal, possibly undertaken as part of a citation cartel for the purpose of gaming the impact factor of the cited journal.', lang='en')],
 'has_label': [Literal('journal cartel citation', lang='en')],
 'is_defined_by': [Model('http://purl.org/spar/cito')],
 'is_sub_concept_of': [Concept('http://purl.org/spar/cito/Citation')]}

And a SKOS concept:

In [8]:
cat = next(c for c in vocabulary.instances(Concept) if c.get_has_identifier().endswith("cat"))

print(cat)
{name: value for name, value in vars(cat).items() if value}

Concept('http://example.org/animals/cat')


{'has_identifier': 'http://example.org/animals/cat',
 'has_preferred_label': [Literal('cat', lang='en')],
 'is_included_in': [Model('http://example.org/animals/scheme')],
 'is_sub_concept_of': [Concept('http://example.org/animals/animal')]}

`vars` shows the raw fields, so use the getters in your code; `vars` is meant for exploring. The `reader.yaml` of each profile lists the setters used for every RDF predicate.

## 3. Inspect the artefact

`read` returns an `Artefact`: everything extracted from the source, with no reference to the code that read it.

| Member | Content |
|--------|---------|
| `read_as` | Name of the format it was read as. |
| `artefact_iri` | IRI of the documented ontology or concept scheme, or `None`. |
| `entities` | Dictionary of the extracted entities, keyed by IRI. Each value is one instance, or a set when the same IRI is used for more than one kind of entity (punning). |
| `instances(cls=None)` | The extracted entities as a list. With a model class from `lode.models`, only the instances of that class and its subclasses. |
| `get_instance(iri)` | The entities registered under an IRI, or `None`. |
| `graph` | The RDF graph, including the imported modules when requested. |
| `main_graph()` | Only the triples of the loaded artefact, without its imports. |
| `imported_uris` | IRIs of the modules pulled in by imports. |
| `warnings` | Reader warnings, when requested with `warnings=True`. |
| `provenance(entity)` | An `rdflib.Graph` with the triples the entity was built from. Computed on demand. |

### 3.1 Walk the Artefact entities

`instances(cls)` takes a model class from `lode.models`. Subclasses are included; without an argument you get everything.

The model classes depend on the format. In OWL, `Concept` is a class, `Relation` an object property, `Attribute` a datatype property, `Annotation` an annotation property; the others are `Individual` and `Model` (the ontology itself). In SKOS they are `Concept`, `Collection` and `Model` (the concept scheme).

In [9]:
from lode.models import Concept, Relation

# di più info sul model -> ntitis l posto di instncs + link modl

classes = ontology.instances(Concept)
len(classes), len(ontology.instances(Relation))

(20, 101)

Model objects expose their data through getters named after the property. Literals have `get_has_label()`

In [10]:
for concept in classes[:5]:
    print(concept, concept.get_has_label())

Concept('http://purl.org/spar/cito/JournalCartelCitation') [Literal('journal cartel citation', lang='en')]
Concept('http://purl.org/spar/cito/DistantCitation') [Literal('distant citation', lang='en')]
Concept('http://purl.org/spar/cito/SelfCitation') [Literal('self citation', lang='en')]
Concept('http://purl.org/spar/cito/JournalSelfCitation') [Literal('journal self citation', lang='en')]
Concept('http://purl.org/spar/cito/AffiliationSelfCitation') [Literal('affiliation self citation', lang='en')]


`entities` is the dictionary behind `instances()`: it is keyed by IRI, and also holds the literals, which `instances()` leaves out. `get_instance(iri)` looks an entity up by IRI and returns a set: an IRI used for more than one kind of entity (punning) gives several instances.

In [11]:
print(len(ontology.entities), "keys")
for key, value in list(ontology.entities.items())[:3]:
    print(key, "->", value)

902 keys
http://purl.org/spar/cito/hasCitingEntity -> {Relation('http://purl.org/spar/cito/hasCitingEntity')}
http://purl.org/spar/cito/hasCitedEntity -> {Relation('http://purl.org/spar/cito/hasCitedEntity')}
http://purl.org/spar/cito/hasCitationCharacterization -> {Relation('http://purl.org/spar/cito/hasCitationCharacterization')}


In [12]:
# normal instance
ontology.get_instance("http://purl.org/spar/cito/cites")

{Relation('http://purl.org/spar/cito/cites')}

### 3.2 The graph

`graph` is the RDF graph the entities were extracted from. `main_graph()` returns only the triples of the loaded artefact, without its imports. Without imports, `main_graph()` returns `graph` itself. With imports it is a view on the named graph of the artefact: the triples are stored once, not copied.

In [13]:
print(type(ontology.graph).__name__, len(ontology.graph), len(ontology.main_graph()))

Graph 988 988


### 3.3 Imports

An ontology can import other ontologies with `owl:imports`. By default only the ontology itself is read. Three switches load the imports as well:

| Switch | Loads |
|--------|-------|
| `imported=True` | The direct imports (depth 1). |
| `closure=True` | The full closure: imports of imports, recursively. |
| `partial_import=True` | Only the entities of the imports that the ontology actually uses. |

With imports, `graph` becomes a `ProvenanceGraph`: a dataset with one named graph per loaded document, named after its ontology IRI, so every triple keeps the document it comes from. It also has an empty default graph. `main_graph()` is the named graph of the artefact alone, and `imported_uris` lists the imported documents.

In [14]:
with_imports = OwlOntology(source, imported=True)

print(type(with_imports.graph).__name__)
print("imported_uris:", with_imports.imported_uris)
for named in with_imports.graph.graphs():
    print(f"{len(named):6} triples  {named.identifier}")
print("union:      ", len(with_imports.graph))
print("main_graph: ", len(with_imports.main_graph()))

ProvenanceGraph
imported_uris: {'http://www.ontologydesignpatterns.org/cp/owl/situation.owl'}
    42 triples  http://www.ontologydesignpatterns.org/cp/owl/situation.owl
   988 triples  http://purl.org/spar/cito
     0 triples  urn:x-rdflib:default
union:       1020
main_graph:  988


### 3.4 Warnings

With `warnings=True` the reader collects its warnings in `warnings`. The current checks are OWL-specific. The list is empty when nothing is wrong.

In [15]:
checked = OwlOntology(source, warnings=True)

print(len(checked.warnings), "warnings")
for warning in checked.warnings:
    print(warning)

0 warnings


## 4. RDF views of an artefact

LODE gives you the RDF of an artefact in four ways. They differ in how much of the graph they cover and in whether they say where each triple comes from:

| View | What it is | How to get it | Type |
|------|------------|---------------|------|
| Whole artefact | All the triples of the loaded artefact, without its imports. This is what LODE exports. | `viewer.export_main(fmt)`, or `artefact.main_graph()` | text, or `Graph` |
| Entity, triples | The triples one entity (its card) is built from. | `artefact.provenance(entity)` | `Graph` |
| Entity, quads | The same triples, each in the named graph of the document it comes from. | `artefact.with_provenance(triples)` | `ProvenanceGraph` |
| Entity, export | The triples of one resource, limited to the loaded artefact. | `viewer.export_resource(iri, fmt)` | text |

Triple formats: `turtle`, `xml`, `json-ld`, `nt` (listed in `lode.helpers.formats.SERIALIZATION_FORMATS`). Quad formats: `trig`, `json-ld`, `nquads`. A format that fails does not raise: you get a text starting with `# serialization error`.

### 4.1 The whole artefact

In [16]:
print(len(ontology.main_graph()), "triples")
print(ontology.viewer.export_main("turtle")[:400])

988 triples
@prefix bibo: <http://purl.org/ontology/bibo/> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema1: <http://schema.org/> .
@pref


### 4.2 One entity: the triples it is built from

`provenance(entity)` is a function, computed on demand. It returns the triples the entity was built from: the ones where it is the subject, those of the blank nodes they lead to, and the statements about it.

In [17]:
(cites,) = ontology.get_instance("http://purl.org/spar/cito/cites")   # one kind of entity, so one instance

triples = ontology.provenance(cites)
print(len(triples), "triples")
print(triples.serialize(format="turtle"))

7 triples
@prefix ns1: <http://www.w3.org/2002/07/owl#> .
@prefix ns2: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ns3: <http://purl.org/vocab/vann/> .
@prefix ns4: <http://www.w3.org/2003/06/sw-vocab-status/ns#> .

<http://purl.org/spar/cito/cites> a ns1:ObjectProperty ;
    ns2:label "cites"@en ;
    ns3:example """
    :paper-a a fabio:JournalArticle ;
        cito:cites :paper-b .
    """ ;
    ns2:comment "A relation according to which a citing entity cites a cited entity, either directly and explicitly (as in the reference list of a journal article), indirectly (e.g. by citing a more recent paper by the same group on the same topic), or implicitly (e.g. as in artistic quotations or parodies, or in cases of plagiarism)."@en ;
    ns2:isDefinedBy <http://purl.org/spar/cito> ;
    ns1:inverseOf <http://purl.org/spar/cito/isCitedBy> ;
    ns4:term_status <http://purl.org/linked-data/registry#statusStable> .




### 4.3 One entity: the same triples as quads

With imports loaded, each triple of an entity can come from a different document. `with_provenance` turns the triples into quads, putting each one in the named graph of its document. This is what the HTML page of an entity offers as TriG, JSON-LD and N-Quads. It is meant for the viewers, so it is less stable than the other views.

In [18]:
(cites,) = with_imports.get_instance("http://purl.org/spar/cito/cites")

quads = with_imports.with_provenance(with_imports.provenance(cites))
print(type(quads).__name__, len(list(quads.quads())), "quads")
print(quads.serialize(format="nquads"))

ProvenanceGraph 7 quads
<http://purl.org/spar/cito/cites> <http://www.w3.org/2002/07/owl#inverseOf> <http://purl.org/spar/cito/isCitedBy> <http://purl.org/spar/cito> .
<http://purl.org/spar/cito/cites> <http://www.w3.org/2000/01/rdf-schema#comment> "A relation according to which a citing entity cites a cited entity, either directly and explicitly (as in the reference list of a journal article), indirectly (e.g. by citing a more recent paper by the same group on the same topic), or implicitly (e.g. as in artistic quotations or parodies, or in cases of plagiarism)."@en <http://purl.org/spar/cito> .
<http://purl.org/spar/cito/cites> <http://www.w3.org/2000/01/rdf-schema#label> "cites"@en <http://purl.org/spar/cito> .
<http://purl.org/spar/cito/cites> <http://purl.org/vocab/vann/example> "\n    :paper-a a fabio:JournalArticle ;\n        cito:cites :paper-b .\n    " <http://purl.org/spar/cito> .
<http://purl.org/spar/cito/cites> <http://www.w3.org/2003/06/sw-vocab-status/ns#term_status> <ht

### 4.4 One resource, as an export

`export_resource` serializes the triples of one resource and keeps only those of the loaded artefact, so the imports never appear in it.

In [19]:
print(ontology.viewer.export_resource("http://purl.org/spar/cito/cites", "turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix vann: <http://purl.org/vocab/vann/> .
@prefix vs: <http://www.w3.org/2003/06/sw-vocab-status/ns#> .

<http://purl.org/spar/cito/cites> a owl:ObjectProperty ;
    rdfs:label "cites"@en ;
    vann:example """
    :paper-a a fabio:JournalArticle ;
        cito:cites :paper-b .
    """ ;
    rdfs:comment "A relation according to which a citing entity cites a cited entity, either directly and explicitly (as in the reference list of a journal article), indirectly (e.g. by citing a more recent paper by the same group on the same topic), or implicitly (e.g. as in artistic quotations or parodies, or in cases of plagiarism)."@en ;
    rdfs:isDefinedBy <http://purl.org/spar/cito> ;
    owl:inverseOf <http://purl.org/spar/cito/isCitedBy> ;
    vs:term_status <http://purl.org/linked-data/registry#statusStable> .




## 5. Build a static site

Same output as `lode build`: `index.html`, the serializations and one page per resource. Existing files with the same names are overwritten.

In [20]:
import tempfile

out = Path(tempfile.mkdtemp(prefix="lode_"))
lode.build_site(ontology, out)
sorted(p.name for p in out.iterdir())[:10]

['index.html',
 'index.jsonld',
 'index.nt',
 'index.ttl',
 'index.xml',
 'resources',
 'static']

## Next

Reference: `docs/usage/python_api.md`. Adding a format: `docs/extending.md`.